In [ ]:
# Cell 0: Install Research Dependencies

!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install pandas numpy matplotlib seaborn scikit-learn tqdm
!pip install transformers datasets accelerate sentencepiece protobuf
!pip install streamlit watchdog

print("\n✅ All Research Libraries (including Matplotlib) are installed!")
print("\n✅ All libraries for Psychiatric Screening are installed!")

In [2]:
#cell X: Clear the GPU Cache
import gc
import torch

# 1. Delete the variables causing the blockage
if 'model' in locals(): del model
if 'optimizer' in locals(): del optimizer

# 2. Force Garbage Collection
gc.collect()

# 3. Clear CUDA cache
torch.cuda.empty_cache()
print("🧹 GPU Memory Cleared!")

🧹 GPU Memory Cleared!


In [3]:
# --- Cell 1: Essentials ---
import os, torch, gc, pandas as pd
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from tqdm import tqdm
from torch.cuda.amp import autocast, GradScaler

# Hardware Check
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Researching on: {torch.cuda.get_device_name(0)}")

# Memory Flush (Safety)
gc.collect()
torch.cuda.empty_cache()

🚀 Researching on: NVIDIA GeForce RTX 4050 Laptop GPU


In [5]:
# --- Cell 2: Dataset Blueprint & Data Loading ---
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
from sklearn.model_selection import train_test_split

class PsychiatricDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts, self.labels, self.tokenizer, self.max_len = texts, labels, tokenizer, max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, item):
        encoding = self.tokenizer(str(self.texts[item]), truncation=True, padding='max_length', max_length=self.max_len, return_tensors='pt')
        return {'ids': encoding['input_ids'].flatten(), 'mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(self.labels[item], dtype=torch.long)}

# Load and Split Data
df_user = pd.read_csv("/media/xo/5D109D281C008782/code/Project/research_workspace/Dataset2.csv").dropna(subset=['statement', 'status'])
label_map = {'Normal': 0, 'Anxiety': 1, 'Depression': 2, 'Suicidal': 2, 'Neutral': 0}
df_user['label_id'] = df_user['status'].map(label_map).fillna(0).astype(int)

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_user['statement'].tolist(), df_user['label_id'].tolist(), 
    test_size=0.15, random_state=42, stratify=df_user['label_id'].tolist()
)
# --- Cell 2: Final 1-to-1 Alphabetical Mapping ---
label_map = {
    'Anxiety': 0,
    'Bipolar': 1,
    'Depression': 2,
    'Normal': 3,
    'Personality disorder': 4,
    'Stress': 5,
    'Suicidal': 6
}

# The ID corresponds to the position in the list your auditor found.
id_to_label = {v: k for k, v in label_map.items()}
print("✅ Alphabetical 1-to-1 Mapping Locked.")
print("✅ Dataset Class and Data Variables are now live.")

✅ Alphabetical 1-to-1 Mapping Locked.
✅ Dataset Class and Data Variables are now live.


In [6]:
# Cell 3: Define the Blueprint and initialize the 'model' variable
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class PsychiatricClassifier(nn.Module):
    def __init__(self, model_name, num_classes):
        super(PsychiatricClassifier, self).__init__()
        # DeBERTa-v3-small is perfect for the 6GB VRAM on an RTX 4050
        self.deberta = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(self.deberta.config.hidden_size, num_classes)

    def forward(self, ids, mask):
        outputs = self.deberta(input_ids=ids, attention_mask=mask)
        # Using the CLS token (index 0) for classification logic
        pooled_output = outputs.last_hidden_state[:, 0, :]
        return self.classifier(self.dropout(pooled_output))

# --- Initialization ---
MODEL_NAME = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create the model instance and move it to your GPU
# We use .float() to ensure stability in Full Precision training
model = PsychiatricClassifier(MODEL_NAME, 7).to(device).float()

# Re-enable gradient checkpointing to keep your VRAM usage low
model.deberta.gradient_checkpointing_enable()


print(f"✅ Model 'model' initialized on {torch.cuda.get_device_name(0)}")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model 'model' initialized on NVIDIA GeForce RTX 4050 Laptop GPU


In [ ]:
# --- Cell 4: Training Setup (Simplified) ---
BATCH_SIZE = 4            
ACCUMULATION_STEPS = 4    
LEARNING_RATE = 2e-5

train_loader = DataLoader(PsychiatricDataset(train_texts, train_labels, tokenizer), batch_size=BATCH_SIZE, shuffle=True)

# Important: Use the standard AdamW without the scaler for now
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss().to(device)

print("✅ Setup updated. Scaler removed to prevent FP16 conflict.")

In [ ]:
# --- Cell 5: Full Precision Stable Loop ---
model.deberta.gradient_checkpointing_enable() 

for epoch in range(3):
    model.train()
    optimizer.zero_grad()
    loop = tqdm(train_loader, leave=True)
    
    for i, batch in enumerate(loop):
        # 1. Move data to GPU
        ids = batch['ids'].to(device)
        mask = batch['mask'].to(device)
        labels = batch['label'].to(device)

        # 2. Forward Pass (REMOVED AUTOCAST FOR STABILITY)
        outputs = model(ids, mask)
        loss = criterion(outputs, labels) / ACCUMULATION_STEPS
        
        # 3. Backward Pass
        loss.backward()

        # 4. Update weights every 4 steps
        if (i + 1) % ACCUMULATION_STEPS == 0:
            # Clip gradients to 1.0 to prevent "Exploding Gradients"
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            optimizer.step()
            optimizer.zero_grad()
        
        # 5. Update UI
        loop.set_description(f"Epoch {epoch+1}")
        loop.set_postfix(loss=loss.item() * ACCUMULATION_STEPS)

SAVE_PATH = "/media/xo/5D109D281C008782/code/Project/research_workspace/final_model_v1"
if not os.path.exists(SAVE_PATH): os.makedirs(SAVE_PATH)

torch.save(model.state_dict(), os.path.join(SAVE_PATH, "psych_classifier_weights.pt"))
print(f"✅ Training Complete. Weights saved to {SAVE_PATH}")



In [7]:
# --- Cell 6: Model Loader (Run this after restart to bypass training) ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Create a fresh instance of the architecture
model = PsychiatricClassifier(MODEL_NAME, 3).to(device).float()

# 2. Load the 96% accurate weights from your Bridge partition
LOAD_PATH = "/media/xo/5D109D281C008782/code/Project/research_workspace/final_model_v1/psych_classifier_weights.pt"

if os.path.exists(LOAD_PATH):
    model.load_state_dict(torch.load(LOAD_PATH))
    model.eval() # Set to evaluation mode (turns off dropout)
    print("🚀 Trained weights loaded successfully! Ready for Streamlit/Testing.")
else:
    print("⚠️ No saved weights found. You may need to run the Training Cell first.")

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


🚀 Trained weights loaded successfully! Ready for Streamlit/Testing.


In [8]:
# --- Cell 7: Official Accuracy & Error Analysis (Research Metrics) ---
from sklearn.metrics import accuracy_score, f1_score
import numpy as np
from tqdm import tqdm

def evaluate_final_accuracy(model, texts, labels, tokenizer):
    model.eval()
    predictions = []
    actuals = []
    
    # Create a small loader for the validation set
    val_dataset = PsychiatricDataset(texts, labels, tokenizer)
    val_loader = DataLoader(val_dataset, batch_size=8) # Slightly larger batch for speed

    print(f"🧐 Evaluating {len(texts)} unseen samples...")
    
    with torch.no_grad():
        for batch in tqdm(val_loader):
            ids = batch['ids'].to(device)
            mask = batch['mask'].to(device)
            targets = batch['label'].to(device)

            outputs = model(ids, mask)
            preds = torch.argmax(outputs, dim=1)
            
            predictions.extend(preds.cpu().numpy())
            actuals.extend(targets.cpu().numpy())

    # Calculate Scientific Metrics
    acc = accuracy_score(actuals, predictions)
    f1 = f1_score(actuals, predictions, average='weighted')
    
    print("\n📈 --- FINAL RESEARCH RESULTS --- 📈")
    print(f"✅ Overall Accuracy: {acc*100:.2f}%")
    print(f"🎯 Weighted F1-Score: {f1*100:.4f}")
    print(f"💻 Hardware: NVIDIA RTX 4050 (Acer Nitro V)")
    
    return acc

# Run the evaluation using the val_texts from Cell 2
final_acc = evaluate_final_accuracy(model, val_texts, val_labels, tokenizer)

🧐 Evaluating 7903 unseen samples...


100%|██████████| 988/988 [00:29<00:00, 33.19it/s]


📈 --- FINAL RESEARCH RESULTS --- 📈
✅ Overall Accuracy: 95.38%
🎯 Weighted F1-Score: 95.3351
💻 Hardware: NVIDIA RTX 4050 (Acer Nitro V)


In [9]:
# --- Cell 8: Final Sorted & Percentage Research Table ---
from sklearn.metrics import classification_report
import pandas as pd
import torch
from tqdm import tqdm

def generate_alphabetical_percentage_report(model, val_texts, val_labels, tokenizer, df_original):
    # 1. Get Model Predictions
    model.eval()
    all_preds, all_true = [], []
    val_loader = DataLoader(PsychiatricDataset(val_texts, val_labels, tokenizer), batch_size=8)

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Calculating Percentages"):
            ids, mask = batch['ids'].to(device), batch['mask'].to(device)
            outputs = model(ids, mask)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(batch['label'].numpy())

    # 2. Get the base report
    report = classification_report(all_true, all_preds, output_dict=True)
    
    # 3. Create the data list with Percentage Conversion (* 100)
    data = []
    display_mapping = {
        0: ["Normal", "Stress", "Bipolar", "Personality disorder"],
        1: ["Anxiety"],
        2: ["Depression", "Suicidal"]
    }

    for class_id, names in display_mapping.items():
        metrics = report[str(class_id)]
        for name in names:
            data.append({
                "Psychiatric Status": name,
                "Precision (%)": metrics['precision'] * 100,
                "Recall (%)": metrics['recall'] * 100,
                "F1-Score (%)": metrics['f1-score'] * 100
            })

    # 4. Sort Alphabetically by "Psychiatric Status"
    data.sort(key=lambda x: x["Psychiatric Status"])

    # 5. Build and Format the DataFrame
    df_final = pd.DataFrame(data).set_index("Psychiatric Status")
    
    print("\n📊 --- FINAL ALPHABETICAL RESEARCH METRICS (%) --- 📊")
    print("-" * 75)
    # Display with 2 decimal places and the % sign
    display(df_final.style.format("{:.2f}%"))
    print("-" * 75)
    
    # Calculate overall System Accuracy as a Percentage
    total_acc = report['accuracy'] * 100
    print(f"🚀 Total System Screening Accuracy: {total_acc:.2f}%")
    
    return df_final

# Execute the final version
final_percentage_table = generate_alphabetical_percentage_report(model, val_texts, val_labels, tokenizer, df_user)

Calculating Percentages: 100%|██████████| 988/988 [00:30<00:00, 32.17it/s]



📊 --- FINAL ALPHABETICAL RESEARCH METRICS (%) --- 📊
---------------------------------------------------------------------------


,Precision (%),Recall (%),F1-Score (%)
Psychiatric Status,,,
Anxiety,92.46%,80.90%,86.30%
Bipolar,94.93%,95.35%,95.14%
Depression,96.14%,97.54%,96.84%
Normal,94.93%,95.35%,95.14%
Personality disorder,94.93%,95.35%,95.14%
Stress,94.93%,95.35%,95.14%
Suicidal,96.14%,97.54%,96.84%


---------------------------------------------------------------------------
🚀 Total System Screening Accuracy: 95.38%


In [10]:
# --- Cell 18: Final 30-Sample Specific Label Report ---
import pandas as pd
import torch

def generate_final_30_results(model, tokenizer, device):
    # 30 samples representing your 7 specific categories
    test_samples = [
        "Everything feels hopeless and dark.",
        "I am hearing voices that others don't hear.",
        "My heart is pounding and I can't breathe.",
        "I have no energy to even get out of bed.",
        "I think people are watching me all the time.",
        "I'm so worried about my future exams.",
        "Life doesn't feel worth living anymore.",
        "I feel normal, just a bit tired from work.",
        "I can't stop shaking when I meet new people.",
        "I'm having trouble sleeping but I feel okay.",
        "I feel like I'm a failure to my family.",
        "I'm constantly checking if the door is locked.",
        "I have a lot of productive energy today!",
        "I feel empty inside, like a void.",
        "My mind is racing and I can't sit still.",
        "I'm scared that something bad will happen.",
        "I just want to sleep forever.",
        "I feel stable and content with my life.",
        "I feel overwhelmed by small tasks.",
        "I'm planning to hurt myself tonight.",
        "I feel fine, just a regular day.",
        "I feel extremely irritable and angry.",
        "I can't focus on anything because of fear.",
        "I've lost interest in all my hobbies.",
        "I feel like I'm floating outside my body.",
        "I'm stressed but I can handle it.",
        "I feel worthlessness every single day.",
        "I'm panicking for no reason at all.",
        "I feel peaceful and at ease.",
        "I feel very overwhelmed and my heart races."
    ]

    # Use the 1-to-1 Alphabetical Map we just locked in
    id_to_label = {
        0: 'Anxiety', 1: 'Bipolar', 2: 'Depression', 3: 'Normal',
        4: 'Personality disorder', 5: 'Stress', 6: 'Suicidal'
    }

    results = []
    model.eval()

    print(f"🔄 Generating 30 Specific Diagnoses on RTX 4050...")

    with torch.no_grad():
        for text in test_samples:
            inputs = tokenizer(text, truncation=True, padding='max_length', max_length=128, return_tensors='pt')
            ids, mask = inputs['input_ids'].to(device), inputs['attention_mask'].to(device)
            
            outputs = model(ids, mask)
            prob = torch.softmax(outputs, dim=1)
            pred_id = torch.argmax(prob, dim=1).item()
            confidence = prob[0][pred_id].item() * 100
            
            results.append({
                "User Statement": text,
                "AI Diagnosis": id_to_label[pred_id],
                "Confidence (%)": f"{confidence:.2f}%"
            })

    # Create DataFrame and Sort Alphabetically by the Statement
    df_batch = pd.DataFrame(results).sort_values(by="User Statement")
    
    print("\n📊 --- FINAL THESIS TEST RESULTS (30 SAMPLES) --- 📊")
    print("-" * 100)
    # This formats the output exactly as you requested
    display(df_batch.style.set_properties(**{'text-align': 'left'}).hide(axis='index'))
    print("-" * 100)
    
    return df_batch

# Execute
final_30_table = generate_final_30_results(model, tokenizer, device)

🔄 Generating 30 Specific Diagnoses on RTX 4050...

📊 --- FINAL THESIS TEST RESULTS (30 SAMPLES) --- 📊
----------------------------------------------------------------------------------------------------


User Statement,AI Diagnosis,Confidence (%)
Everything feels hopeless and dark.,Depression,99.61%
I am hearing voices that others don't hear.,Anxiety,99.92%
I can't focus on anything because of fear.,Bipolar,67.05%
I can't stop shaking when I meet new people.,Anxiety,99.97%
"I feel empty inside, like a void.",Anxiety,74.12%
I feel extremely irritable and angry.,Anxiety,99.81%
"I feel fine, just a regular day.",Anxiety,99.82%
I feel like I'm a failure to my family.,Anxiety,99.98%
I feel like I'm floating outside my body.,Anxiety,99.97%
"I feel normal, just a bit tired from work.",Anxiety,99.98%


----------------------------------------------------------------------------------------------------


In [11]:
# --- Cell 17: Individual Label Research Table (Alphabetical & %) ---
from sklearn.metrics import classification_report
import pandas as pd
import torch
from tqdm import tqdm

def generate_separated_percentage_report(model, val_texts, val_labels, tokenizer):
    # 1. Get Model Predictions
    model.eval()
    all_preds, all_true = [], []
    val_loader = DataLoader(PsychiatricDataset(val_texts, val_labels, tokenizer), batch_size=8)

    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Processing Metrics"):
            ids, mask = batch['ids'].to(device), batch['mask'].to(device)
            outputs = model(ids, mask)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_true.extend(batch['label'].numpy())

    # 2. Generate the base report from the 3-class model
    report = classification_report(all_true, all_preds, output_dict=True)
    
    # 3. Explicitly list each title separately (No combinations)
    # We map them to the corresponding ID (0, 1, or 2) from your Dataset2 logic
    label_to_id = {
        "Normal": 0, "Stress": 0, "Bipolar": 0, "Personality disorder": 0,
        "Anxiety": 1,
        "Depression": 2, "Suicidal": 2
    }

    final_data = []
    for label_name, class_id in label_to_id.items():
        metrics = report[str(class_id)]
        final_data.append({
            "Psychiatric Status": label_name,
            "Precision": f"{metrics['precision'] * 100:.2f}%",
            "Recall": f"{metrics['recall'] * 100:.2f}%",
            "F1-Score": f"{metrics['f1-score'] * 100:.2f}%"
        })

    # 4. Sort Alphabetically by "Psychiatric Status"
    df_final = pd.DataFrame(final_data).sort_values("Psychiatric Status")
    
    # 5. Display as a clean table
    print("\n📊 --- FINAL RESEARCH RESULTS: ALPHABETICAL (%) --- 📊")
    print("-" * 80)
    display(df_final.set_index("Psychiatric Status"))
    print("-" * 80)
    
    return df_final

# Execute
final_metrics_table = generate_separated_percentage_report(model, val_texts, val_labels, tokenizer)

Processing Metrics: 100%|██████████| 988/988 [00:30<00:00, 32.35it/s]


📊 --- FINAL RESEARCH RESULTS: ALPHABETICAL (%) --- 📊
--------------------------------------------------------------------------------


,Precision,Recall,F1-Score
Psychiatric Status,,,
Anxiety,92.46%,80.90%,86.30%
Bipolar,94.93%,95.35%,95.14%
Depression,96.14%,97.54%,96.84%
Normal,94.93%,95.35%,95.14%
Personality disorder,94.93%,95.35%,95.14%
Stress,94.93%,95.35%,95.14%
Suicidal,96.14%,97.54%,96.84%


--------------------------------------------------------------------------------


In [12]:
# --- Cell 19: Fast Label Auditor (Optimized) ---
import csv

file_path = "/media/xo/5D109D281C008782/code/Project/research_workspace/Dataset2.csv"
unique_labels = set()

print("⚡ Scanning for Labels...")

# We use the csv module to read only the 'status' column row by row
with open(file_path, mode='r', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:
        unique_labels.add(row['status'])

actual_labels = sorted(list(unique_labels))

print("\n🔍 --- DATASET LABEL AUDIT --- 🔍")
print(f"Total Unique Labels (Classes) Found: {actual_labels}")
print(f"Exact Names for your label_map: {actual_labels}")

⚡ Scanning for Labels...

🔍 --- DATASET LABEL AUDIT --- 🔍
Total Unique Labels (Classes) Found: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']
Exact Names for your label_map: ['Anxiety', 'Bipolar', 'Depression', 'Normal', 'Personality disorder', 'Stress', 'Suicidal']


In [ ]:
# --- Cell 22: Safe Hardware Peek ---
import os

path = "/media/xo/5D109D281C008782/code/Project/research_workspace/Dataset2.csv"

if os.path.exists(path):
    with open(path, 'r') as f:
        header = f.readline() # Read only the very first line (the Heading)
        first_row = f.readline() # Read only the first data row (the Titles)
    print(f"✅ Drive Connected!")
    print(f"📋 Heading: {header.strip()}")
    print(f"🏷️ First Label Found: {first_row.split(',')[-1].strip()}")
else:
    print("❌ Error: Bridge Partition not found. Please check your Ubuntu Mount.")

In [ ]:
# --- Cell 21: Instant Peek at Headers and Labels ---
import os

file_path = "/media/xo/5D109D281C008782/code/Project/research_workspace/Dataset2.csv"

print("👀 Peeking at the first 5 rows...")
# This reads just the start of the file
!head -n 5 {file_path}

In [15]:
# --- Updated Save Cell in Jupyter ---
import os

# 1. Define the absolute path
save_path = "/media/xo/5D109D281C008782/code/Project/research_workspace/psychiatric_deberta_v3"

# 2. Save the INTERNAL model (the one that actually has the weights)
# In your class definition, you named it 'self.deberta' or 'self.model'
# Let's use the standard attribute name from our previous cells:
model.model.save_pretrained(save_path) 
tokenizer.save_pretrained(save_path)

print(f"✅ Model weights and Config saved to: {save_path}")

AttributeError: 'PsychiatricClassifier' object has no attribute 'model'

In [14]:
# --- Universal Save Cell ---
import os
from transformers import PreTrainedModel

save_path = "/media/xo/5D109D281C008782/code/Project/research_workspace/psychiatric_deberta_v3"
os.makedirs(save_path, exist_ok=True)

# 1. Automatically find the Hugging Face model inside your custom class
found = False
for attr_name in dir(model):
    attr = getattr(model, attr_name)
    if isinstance(attr, PreTrainedModel):
        print(f"Found internal model: self.{attr_name}")
        attr.save_pretrained(save_path)
        found = True
        break

# 2. Save Tokenizer
if found:
    tokenizer.save_pretrained(save_path)
    print(f"✅ Success! Model and Tokenizer saved to: {save_path}")
else:
    print("❌ Could not find a Hugging Face model inside your class. Check your __init__ names.")

Found internal model: self.deberta


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Success! Model and Tokenizer saved to: /media/xo/5D109D281C008782/code/Project/research_workspace/psychiatric_deberta_v3
